#### The target df is missing some required field for the context-aware detection model ("post_title", "parent_text"). We need to merge it with the post and comments dataframes to augment it with contextual metadata.

## Augment target df

#### Import and load files

In [14]:
import pandas as pd
import numpy as np

targets = pd.read_csv("../data/targets/stop_target.csv", low_memory=False)
posts       = pd.read_csv("../data/clean_flag_post.csv",         low_memory=False, sep=",")
comments    = pd.read_csv("../data/clean_df.csv",      low_memory=False)

print("targets:", targets.shape)
print("posts:      ", posts.shape)
print("comments:   ", comments.shape)

targets: (4306, 33)
posts:       (30159, 67)
comments:    (428122, 26)


In [15]:
print("=== targets ===")
print(targets[["post_id", "post_url","comment_id", "in_reply_to"]].dtypes)
print(targets[["post_id", "post_url", "comment_id", "in_reply_to"]].head(3))

print("\n=== posts ===")
print(posts[["id", "post_url"]].dtypes)
print(posts[["id", "post_url"]].head(3))

print("\n=== comments ===")
print(comments[["post_url", "post_id", "comment_id","in_reply_to"]].dtypes)
print(comments[["post_url", "post_id", "comment_id", "in_reply_to"]].head(3))

=== targets ===
post_id        object
post_url       object
comment_id     object
in_reply_to    object
dtype: object
    post_id                                           post_url  \
0  1,71E+15  https://www.facebook.com/152530751460408/posts...   
1  1,71E+15  https://www.facebook.com/152530751460408/posts...   
2  1,71E+15  https://www.facebook.com/152530751460408/posts...   

         comment_id in_reply_to  
0          1,71E+15         NaN  
1  1712985178748283         NaN  
2  1712613882118746         NaN  

=== posts ===
id          object
post_url    object
dtype: object
                                 id  \
0  637517436414919_1302115643288425   
1  898813010289255_1061589347344953   
2  849855428464829_2117554938361532   

                                            post_url  
0  https://www.facebook.com/groups/63751743641491...  
1  https://www.facebook.com/groups/89881301028925...  
2  https://www.facebook.com/groups/84985542846482...  

=== comments ===
post_url       obje

In [16]:
# How many post_ids are in scientific notation in targets?
sci_mask = targets["post_id"].astype(str).str.match(r"^[\d,]+E\+\d+$")
print(f"Targets with scientific-notation post_id: {sci_mask.sum()}")
print(targets.loc[sci_mask, ["post_id", "comment_id", "post_url"]].head(5))

# Same check on comment_id
sci_mask_cid = targets["comment_id"].astype(str).str.match(r"^[\d,]+E\+\d+$")
print(f"\nTargets with scientific-notation comment_id: {sci_mask_cid.sum()}")

Targets with scientific-notation post_id: 224
    post_id        comment_id  \
0  1,71E+15          1,71E+15   
1  1,71E+15  1712985178748283   
2  1,71E+15  1712613882118746   
3  1,71E+15  1712607358786065   
4  1,71E+15  1713543518692449   

                                            post_url  
0  https://www.facebook.com/152530751460408/posts...  
1  https://www.facebook.com/152530751460408/posts...  
2  https://www.facebook.com/152530751460408/posts...  
3  https://www.facebook.com/152530751460408/posts...  
4  https://www.facebook.com/152530751460408/posts...  

Targets with scientific-notation comment_id: 259


In [17]:
import re
import numpy as np
import pandas as pd

def normalise_id(series):
    """Strip whitespace and remove trailing .0 float artifacts."""
    def _fix(val):
        if pd.isna(val):
            return np.nan
        s = str(val).strip()
        s = re.sub(r"\.0$", "", s)
        return s
    return series.apply(_fix)


# ── posts: normalise post_url only ────────────────────────────────────────────
posts["post_url"] = posts["post_url"].astype(str).str.strip().str.rstrip("/")

# ── targets: normalise post_url + comment ids ─────────────────────────────────
targets["post_url"]    = targets["post_url"].astype(str).str.strip().str.rstrip("/")
targets["comment_id"]  = normalise_id(targets["comment_id"])
targets["in_reply_to"] = normalise_id(targets["in_reply_to"])

# ── comments: normalise ids and post_url ─────────────────────────────────────
comments["comment_id"]  = normalise_id(comments["comment_id"])
comments["in_reply_to"] = normalise_id(comments["in_reply_to"])
comments["post_url"]    = comments["post_url"].astype(str).str.strip().str.rstrip("/")


#### join targets and posts

In [18]:
posts_slim = posts[["post_url", "title", "message", "description"]].rename(columns={
    "title":       "post_title",
    "message":     "post_message",
    "description": "post_description",
}).drop_duplicates(subset="post_url")

n_before = len(targets)
targets = targets.merge(posts_slim, on="post_url", how="left")
assert len(targets) == n_before, "Post merge duplicated rows — check for duplicate post_urls in posts file"

# ── Coverage check ─────────────────────────────────────────────────────────
missing_post = targets["post_title"].isna()
print(f"Total rows:                       {len(targets)}")
print(f"Matched to a post:                {(~missing_post).sum()}")
print(f"NOT matched (no post found):      {missing_post.sum()}")
if missing_post.sum() > 0:
    print("\nSample unmatched post_urls:")
    print(targets.loc[missing_post, "post_url"].value_counts().head(10))

Total rows:                       4306
Matched to a post:                3825
NOT matched (no post found):      481

Sample unmatched post_urls:
post_url
https://www.facebook.com/157495094262296/posts/2548931701785278    8
https://www.facebook.com/nonauvaccin/posts/344614212877660         6
https://www.facebook.com/BMFTVNEWS/posts/620649471633159           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633037           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633446           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633287           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633049           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633385           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633333           1
https://www.facebook.com/BMFTVNEWS/posts/620649471633232           1
Name: count, dtype: int64


In [19]:
targets.head(3)

,ordre,url_id,clean_url,parent_domain,source,thématique,sous-thématique,false_news_usr_feedback,hate_speech_usr_feedback,account_name,...,justification,nb_character,comment_lenght,formatted_date,reactions,replies,in_reply_to,post_title,post_message,post_description
0,8958.0,bvpvs04nvswz0rt,http://lesmoutonsrebelles.com/encore-un-bebe-d...,lesmoutonsrebelles.com,contre-information,santé,vaccin,311,-11,"J'arrete de travailler, je boycotte ce systeme...",...,épisémique,"31,00",short,27 avr. 2018,4,1,NaN,Encore un bébé de deux mois qui décède 48H apr...,NaN,La maman du petit Owen s’exprime : Des décès p...
1,8961.0,bvpvs04nvswz0rt,http://lesmoutonsrebelles.com/encore-un-bebe-d...,lesmoutonsrebelles.com,contre-information,santé,vaccin,311,-11,"J'arrete de travailler, je boycotte ce systeme...",...,épisémique,"77,00",medium,27 avr. 2018,4,0,NaN,Encore un bébé de deux mois qui décède 48H apr...,NaN,La maman du petit Owen s’exprime : Des décès p...
2,8981.0,bvpvs04nvswz0rt,http://lesmoutonsrebelles.com/encore-un-bebe-d...,lesmoutonsrebelles.com,contre-information,santé,vaccin,311,-11,"J'arrete de travailler, je boycotte ce systeme...",...,épisémique,"111,00",long,26 avr. 2018,3,0,NaN,Encore un bébé de deux mois qui décède 48H apr...,NaN,La maman du petit Owen s’exprime : Des décès p...


### join with comments

In [22]:
print("=== targets columns ===")
print(targets[["post_url", "text", "user_handle", "user_labl", "user_id", "user_url"]].head(3))

print("\n=== comments columns ===")
print(comments[["post_url", "text", "user_label"]].head(3))

=== targets columns ===
                                            post_url  \
0  https://www.facebook.com/152530751460408/posts...   
1  https://www.facebook.com/152530751460408/posts...   
2  https://www.facebook.com/152530751460408/posts...   

                                                text         user_handle  \
0                    la source n'a rien de fiable...  bertrand.martin.10   
1  Trop de conneries partagées sur cette page, do...             morgauz   
2  En même temps avant les vaccins c était des mi...            ezrec333   

                          user_labl  user_id  \
0  Perceval Le Gaulois Martinmichel      NaN   
1                      Fanny Segret      NaN   
2                   Etienne Zbaeren      NaN   

                                      user_url  
0  https://www.facebook.com/bertrand.martin.10  
1             https://www.facebook.com/morgauz  
2            https://www.facebook.com/ezrec333  

=== comments columns ===
                               

In [23]:
# How many post_urls actually overlap between targets and comments?
target_urls  = set(targets["post_url"].str.strip().str.rstrip("/"))
comment_urls = set(comments["post_url"].str.strip().str.rstrip("/"))

print(f"Unique post_urls in targets:          {len(target_urls)}")
print(f"Unique post_urls in comments:         {len(comment_urls)}")
print(f"Overlap:                              {len(target_urls & comment_urls)}")
print(f"Targets urls NOT in comments:         {len(target_urls - comment_urls)}")

# Sample the non-overlapping ones to understand the format difference
non_overlap = targets[~targets["post_url"].str.strip().str.rstrip("/").isin(comment_urls)]
print("\nSample target post_urls not found in comments:")
print(non_overlap["post_url"].drop_duplicates().head(5).tolist())
print("\nSample comment post_urls:")
print(comments["post_url"].drop_duplicates().head(5).tolist())

# Check user_labl in targets vs user_label in comments
print("\n=== user field check ===")
print("targets user_labl sample:  ", targets["user_labl"].dropna().head(5).tolist())
print("comments user_label sample:", comments["user_label"].dropna().head(5).tolist())

# Overlap on user labels
target_users  = set(targets["user_labl"].dropna().str.strip())
comment_users = set(comments["user_label"].dropna().str.strip())
print(f"\nUnique user_labl in targets:          {len(target_users)}")
print(f"Unique user_label in comments:        {len(comment_users)}")
print(f"Overlap:                              {len(target_users & comment_users)}")

Unique post_urls in targets:          777
Unique post_urls in comments:         6546
Overlap:                              312
Targets urls NOT in comments:         465

Sample target post_urls not found in comments:
['https://www.facebook.com/BMFTVNEWS/posts/620649471633031', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633307', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633315', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633259', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633326']

Sample comment post_urls:
['https://www.facebook.com/groups/637517436414919/permalink/1302115643288425', 'https://www.facebook.com/groups/898813010289255/permalink/1061589347344953', 'https://www.facebook.com/groups/849855428464829/permalink/2117554938361532', 'https://www.facebook.com/groups/99815155485/permalink/10161950464175486', 'https://www.facebook.com/groups/428956983966519/permalink/1028223214039890']

=== user field check ===
targets user_labl sample:   ['Perceval Le

In [24]:
import re

def extract_post_id_from_url(url):
    """Extract the last long numeric segment from any FB post URL format."""
    if not isinstance(url, str):
        return None
    m = re.search(r"/(\d{8,})[/?]?$", url.rstrip("/"))
    return m.group(1) if m else None

targets["_post_id_from_url"]  = targets["post_url"].apply(extract_post_id_from_url)
comments["_post_id_from_url"] = comments["post_url"].apply(extract_post_id_from_url)

target_pids  = set(targets["_post_id_from_url"].dropna())
comment_pids = set(comments["_post_id_from_url"].dropna())

print(f"Unique post ids extracted from targets urls:   {len(target_pids)}")
print(f"Unique post ids extracted from comments urls:  {len(comment_pids)}")
print(f"Overlap on extracted post id:                  {len(target_pids & comment_pids)}")
print(f"Targets still not matched via post id:         {len(target_pids - comment_pids)}")

# Show sample of still-unmatched
still_out = targets[~targets["_post_id_from_url"].isin(comment_pids)]
print("\nSample post_urls with no match even on extracted post id:")
print(still_out["post_url"].drop_duplicates().head(5).tolist())

Unique post ids extracted from targets urls:   777
Unique post ids extracted from comments urls:  6546
Overlap on extracted post id:                  312
Targets still not matched via post id:         465

Sample post_urls with no match even on extracted post id:
['https://www.facebook.com/BMFTVNEWS/posts/620649471633031', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633307', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633315', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633259', 'https://www.facebook.com/BMFTVNEWS/posts/620649471633326']


### build composite key on user_label + text (since the post_url are not matching for enough comments)

In [25]:
import hashlib

def make_key(text, user):
    """Stable hash of stripped+lowercased text + user label."""
    combined = "||".join([
        str(text or "").strip().lower(),
        str(user or "").strip().lower(),
    ])
    return hashlib.md5(combined.encode()).hexdigest()

# Build key on comments side: this is our lookup table
# user_label in comments matches user_labl in targets
comments["_key"] = comments.apply(
    lambda r: make_key(r["text"], r["user_label"]), axis=1
)

# Check key uniqueness in comments — duplicates would cause wrong parent_text matches
n_dupes = comments["_key"].duplicated().sum()
print(f"Duplicate keys in comments: {n_dupes} / {len(comments)}")
if n_dupes > 0:
    dupe_mask = comments["_key"].duplicated(keep=False)
    print("Sample duplicate keys:")
    print(comments.loc[dupe_mask, ["text", "user_label", "_key"]].head(6))

# Build reverse lookup: key → comment_id
key_to_comment_id = (
    comments[["_key", "comment_id"]]
    .drop_duplicates(subset="_key", keep="first")
    .set_index("_key")["comment_id"]
    .to_dict()
)
print(f"\nLookup table size: {len(key_to_comment_id)} unique keys")

Duplicate keys in comments: 16872 / 428122
Sample duplicate keys:
                                                  text            user_label  \
70                                      Si tu le dis..  Séverine Prune Bloch   
114  E naddition qu'il n'y a pas de Frontières dans...      Patrick Douville   
123                                                Ptg        Bruno Kirchner   
127                                                Ptg        Bruno Kirchner   
229                                Et on lui fera rien        Therese Andreu   
410  La Suede a été le premier pays perdu de l'Europe.      Soutien Français   

                                 _key  
70   0ec1ec02420e3e0d1a1e67da780bdfb5  
114  63282956da171ebdacfa233cfe69605e  
123  6b1308ee8dd30a234dd95c8a4e758c79  
127  6b1308ee8dd30a234dd95c8a4e758c79  
229  d95a90c8888101fb609f528880ee9c43  
410  f8f5a131eae6a0cc02b5213ca29ba64d  

Lookup table size: 411250 unique keys


In [26]:
# Build key on targets side using their own text + user_labl
targets["_key"] = targets.apply(
    lambda r: make_key(r["text"], r["user_labl"]), axis=1
)

# Identify corrupted/missing comment_ids
sci_mask    = targets["comment_id"].astype(str).str.match(r"^[\d.,]+E[+\-]\d+$", case=False)
null_mask   = targets["comment_id"].isna()
corrupted   = sci_mask | null_mask
print(f"Corrupted/missing comment_ids: {corrupted.sum()}")

# Recover via composite key
targets["comment_id_recovered"] = targets["comment_id"].copy()
targets.loc[corrupted, "comment_id_recovered"] = (
    targets.loc[corrupted, "_key"].map(key_to_comment_id)
)

n_recovered = (corrupted & targets["comment_id_recovered"].notna()).sum()
n_still_bad = (corrupted & targets["comment_id_recovered"].isna()).sum()
print(f"  Recovered via text+user key:   {n_recovered}")
print(f"  Still unresolved:              {n_still_bad}")

# Use recovered id as authoritative
targets["comment_id"] = targets["comment_id_recovered"]

Corrupted/missing comment_ids: 739
  Recovered via text+user key:   258
  Still unresolved:              481


### join with comments df

In [27]:
# Lookup: comment_id → (text, post_url) for parent validation
id_to_parent = (
    comments[["comment_id", "text", "post_url"]]
    .drop_duplicates(subset="comment_id", keep="first")
    .rename(columns={"text": "parent_text", "post_url": "_parent_post_url"})
    .set_index("comment_id")
)

has_parent = targets["in_reply_to"].notna() & (targets["in_reply_to"] != "nan")

targets["parent_text"]      = targets["in_reply_to"].map(id_to_parent["parent_text"])
targets["_parent_post_url"] = targets["in_reply_to"].map(id_to_parent["_parent_post_url"])

# Sanity check: parent should belong to the same post
url_mismatch = (
    has_parent
    & targets["parent_text"].notna()
    & (targets["post_url"].str.rstrip("/") != targets["_parent_post_url"].str.rstrip("/"))
)

print(f"Rows with in_reply_to set:              {has_parent.sum()}")
print(f"  parent_text found:                    {(has_parent & targets['parent_text'].notna()).sum()}")
print(f"  parent_text missing:                  {(has_parent & targets['parent_text'].isna()).sum()}")
print(f"  post_url mismatch (bad joins):        {url_mismatch.sum()}")

# Nullify any mismatch — better to have empty parent_text than a wrong one
if url_mismatch.sum() > 0:
    print("  → Nullifying mismatched parent_text rows")
    targets.loc[url_mismatch, "parent_text"] = np.nan

Rows with in_reply_to set:              1224
  parent_text found:                    1094
  parent_text missing:                  130
  post_url mismatch (bad joins):        386
  → Nullifying mismatched parent_text rows


### Conclusion: after the two joins (posts and comments)

In [28]:
context_cols = {
    "post_title":       "from posts join",
    "post_message":     "from posts join",
    "post_description": "from posts join",
    "parent_text":      "from comments join (reply rows only)",
}

total = len(targets)
has_parent = targets["in_reply_to"].notna() & (targets["in_reply_to"] != "nan")

print("=" * 65)
print(f"{'CONTEXT FIELD COVERAGE':^65}")
print("=" * 65)
print(f"Total annotated rows: {total}")
print(f"Reply rows (in_reply_to set): {has_parent.sum()} "
      f"({100*has_parent.sum()/total:.1f}%)")
print("-" * 65)
print(f"{'Field':<22} {'Non-empty':>10} {'Empty':>8} {'% filled':>10}  Note")
print("-" * 65)

for col, note in context_cols.items():
    if col not in targets.columns:
        print(f"  {col:<20} {'MISSING COLUMN':>30}")
        continue
    non_empty = (targets[col].fillna("").str.strip() != "").sum()
    empty     = total - non_empty
    print(f"  {col:<20} {non_empty:>10} {empty:>8} {100*non_empty/total:>9.1f}%  {note}")

print("-" * 65)

# parent_text broken down by reply vs non-reply rows
if "parent_text" in targets.columns:
    pt_filled_replies    = (has_parent & (targets["parent_text"].fillna("").str.strip() != "")).sum()
    pt_missing_replies   = (has_parent & (targets["parent_text"].fillna("").str.strip() == "")).sum()
    pt_nonreply_nonempty = (~has_parent & (targets["parent_text"].fillna("").str.strip() != "")).sum()
    print(f"\n  parent_text detail:")
    print(f"    reply rows with parent found:    {pt_filled_replies} / {has_parent.sum()} "
          f"({100*pt_filled_replies/max(has_parent.sum(),1):.1f}%)")
    print(f"    reply rows with parent missing:  {pt_missing_replies} / {has_parent.sum()} "
          f"({100*pt_missing_replies/max(has_parent.sum(),1):.1f}%)")
    if pt_nonreply_nonempty > 0:
        print(f"    WARNING: {pt_nonreply_nonempty} non-reply rows have a non-empty parent_text — investigate")

print("=" * 65)

# Rows with ALL context fields filled vs at least one empty
all_filled = targets[list(context_cols.keys())].fillna("").apply(
    lambda col: col.str.strip() != ""
).all(axis=1)
any_filled = targets[list(context_cols.keys())].fillna("").apply(
    lambda col: col.str.strip() != ""
).any(axis=1)

print(f"\nRows with ALL 4 context fields non-empty:  {all_filled.sum()} ({100*all_filled.sum()/total:.1f}%)")
print(f"Rows with AT LEAST 1 context field:        {any_filled.sum()} ({100*any_filled.sum()/total:.1f}%)")
print(f"Rows with NO context at all:               {(~any_filled).sum()} ({100*(~any_filled).sum()/total:.1f}%)")

                     CONTEXT FIELD COVERAGE                      
Total annotated rows: 4306
Reply rows (in_reply_to set): 1224 (28.4%)
-----------------------------------------------------------------
Field                   Non-empty    Empty   % filled  Note
-----------------------------------------------------------------
  post_title                 3825      481      88.8%  from posts join
  post_message               3300     1006      76.6%  from posts join
  post_description           3107     1199      72.2%  from posts join
  parent_text                3790      516      88.0%  from comments join (reply rows only)
-----------------------------------------------------------------

  parent_text detail:
    reply rows with parent found:    708 / 1224 (57.8%)
    reply rows with parent missing:  516 / 1224 (42.2%)

Rows with ALL 4 context fields non-empty:  2472 (57.4%)
Rows with AT LEAST 1 context field:        3920 (91.0%)
Rows with NO context at all:               386 (9.0%)

In [29]:
import os 

out_path = "../data/targets/stop_target_augmented.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
targets.to_csv(out_path, index=False)
print(f"Saved {len(targets)} rows to {out_path}")
print(f"Columns: {targets.columns.tolist()}")

Saved 4306 rows to ../data/targets/stop_target_augmented.csv
Columns: ['ordre', 'url_id', 'clean_url', 'parent_domain', 'source', 'thématique', 'sous-thématique', 'false_news_usr_feedback', 'hate_speech_usr_feedback', 'account_name', 'charte de modération', 'account_subscriber_count', 'audience_size', 'territoire', 'post_url', 'share_title', 'post_id', 'comment_id', 'user_id', 'user_handle', 'user_url', 'user_labl', 'text', 'target', 'target2', 'target3', 'justification', 'nb_character', 'comment_lenght', 'formatted_date', 'reactions', 'replies', 'in_reply_to', 'post_title', 'post_message', 'post_description', '_post_id_from_url', '_key', 'comment_id_recovered', 'parent_text', '_parent_post_url']
